# 02 — Data Cleaning and Preprocessing

**Course:** COMP4381 — Data Science and Analytics  
**Semester:** Spring 2026  

---

## Purpose

This notebook takes the merged raw dataset produced by `01_data_collection.ipynb`
and applies a structured cleaning pipeline to prepare it for exploratory analysis and modelling.

**Steps performed:**

| Step | Description |
|------|-------------|
| 1 | Initial inspection (shape, dtypes, head, describe) |
| 2 | Missing-value analysis and treatment |
| 3 | Duplicate detection and removal |
| 4 | Data-type corrections |
| 5 | Outlier detection and capping (IQR method) |
| 6 | Ordinal encoding for categorical features |
| 7 | Final validation and export |

**Input:** `data/processed/merged_raw.csv` (107,280 rows × 37 columns)  
**Output:** `data/processed/flight_delays_clean.csv`

## 1. Imports and Data Loading

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
INPUT_PATH = PROCESSED_DIR / "merged_raw.csv"
OUTPUT_PATH = PROCESSED_DIR / "flight_delays_clean.csv"

assert INPUT_PATH.exists(), f"Input file not found: {INPUT_PATH}"

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
print(f"Loaded: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Loaded: 107,280 rows  x  37 columns
Memory usage: 95.3 MB


## 2. Initial Inspection

A quick overview of every column: data type, non-null count, and representative values.
This tells us where the problems are before we start fixing them.

In [6]:
df.info()
#To print all information and data

<class 'pandas.DataFrame'>
RangeIndex: 107280 entries, 0 to 107279
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   icao_code             107280 non-null  str           
 1   apt_name              107280 non-null  str           
 2   country               107280 non-null  str           
 3   date                  107280 non-null  datetime64[us]
 4   arrivals              107280 non-null  int64         
 5   total_atfm_delay_min  107280 non-null  int64         
 6   flights_with_delay    107280 non-null  int64         
 7   flights_delayed_15    107280 non-null  int64         
 8   avg_atfm_delay_min    105193 non-null  float64       
 9   pct_delayed_15        105193 non-null  float64       
 10  delay_weather_min     107280 non-null  int64         
 11  delay_capacity_min    107280 non-null  int64         
 12  delay_staffing_min    107280 non-null  int64         
 13  departures

In [3]:
df.head(10)
#Print the first 10 rows of data

,icao_code,apt_name,country,date,arrivals,total_atfm_delay_min,flights_with_delay,flights_delayed_15,avg_atfm_delay_min,pct_delayed_15,...,primary_surface,year,month,quarter,weekday,is_weekend,season,is_summer,hub_flag,traffic_volume_class
0,EGPH,Edinburgh,United Kingdom,2023-01-01,105,77,4,2,0.733333,1.904762,...,ASP,2023,1,1,6,1,winter,0,1,high
1,EDDP,Leipzig-Halle,Germany,2023-01-03,102,116,16,1,1.137255,0.980392,...,CON,2023,1,1,1,0,winter,0,1,high
2,GCTS,Tenerife Sur/Reina Sofia,Spain,2023-01-03,145,686,43,19,4.731034,13.103448,...,ASP,2023,1,1,1,0,winter,0,1,high
3,LFBO,Toulouse-Blagnac,France,2023-01-04,96,163,14,5,1.697917,5.208333,...,ASP,2023,1,1,2,0,winter,0,1,high
4,LFPO,Paris-Orly,France,2023-01-07,248,572,36,11,2.306452,4.435484,...,CON,2023,1,1,5,1,winter,0,1,high
5,EGLL,London/ Heathrow,United Kingdom,2023-01-08,589,1526,80,42,2.590832,7.130730,...,ASP,2023,1,1,6,1,winter,0,1,high
6,LFPG,Paris-Charles-de-Gaulle,France,2023-01-09,581,125,15,2,0.215146,0.344234,...,ASP,2023,1,1,0,0,winter,0,1,high
7,EGKK,London/ Gatwick,United Kingdom,2023-01-11,194,0,0,0,0.000000,0.000000,...,ASP,2023,1,1,2,0,winter,0,1,high
8,LFPO,Paris-Orly,France,2023-01-11,207,144,16,2,0.695652,0.966184,...,CON,2023,1,1,2,0,winter,0,1,high
9,LFLB,Chambéry-Aix-les-Bains,France,2023-01-14,43,845,28,19,19.651163,44.186047,...,ASP,2023,1,1,5,1,winter,0,0,medium


In [4]:
df.describe()
#To show basic statistics for the numerical columns.


,date,arrivals,total_atfm_delay_min,flights_with_delay,flights_delayed_15,avg_atfm_delay_min,pct_delayed_15,delay_weather_min,delay_capacity_min,delay_staffing_min,...,scheduled_service,runway_count,max_runway_length_ft,year,month,quarter,weekday,is_weekend,is_summer,hub_flag
count,107280,107280.000000,107280.000000,107280.000000,107280.000000,105193.000000,105193.000000,107280.000000,107280.000000,107280.000000,...,103466.000000,103466.000000,103466.000000,107280.000000,107280.000000,107280.000000,107280.000000,107280.000000,107280.000000,107280.000000
mean,2023-12-31 20:10:02.416107,74.494202,71.065287,3.550308,1.516322,0.316363,0.640347,38.384620,4.642040,5.883837,...,0.851458,1.649247,8774.674279,2023.500606,6.515613,2.506758,2.969491,0.277069,0.253300,0.265744
min,2023-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,3609.000000,2023.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,2023-07-02 00:00:00,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,7218.000000,2023.000000,4.000000,2.000000,1.000000,0.000000,0.000000,0.000000
50%,2024-01-01 00:00:00,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,8245.000000,2024.000000,7.000000,3.000000,3.000000,0.000000,0.000000,0.000000
75%,2024-07-02 00:00:00,84.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,2.000000,10171.000000,2024.000000,10.000000,4.000000,5.000000,1.000000,1.000000,1.000000
max,2024-12-31 00:00:00,754.000000,30508.000000,509.000000,440.000000,335.000000,300.000000,30508.000000,6138.000000,16908.000000,...,1.000000,6.000000,15807.000000,2024.000000,12.000000,4.000000,6.000000,1.000000,1.000000,1.000000
std,NaN,133.826541,583.353635,20.960943,11.063950,2.871704,3.891286,501.411892,64.502135,140.235416,...,0.355638,0.883149,2339.231336,0.500002,3.436138,1.114905,1.994907,0.447553,0.434903,0.441731


In [5]:
df.describe(include="str")
#To get statistics for text columns

,icao_code,apt_name,country,type,name,iso_country,iso_region,municipality,iata_code,primary_surface,season,traffic_volume_class
count,107280,107280,107280,103466,103466,103466,103466,103466,100560,103466,107280,107280
unique,155,155,5,2,147,5,49,140,142,8,4,3
top,EGPH,Edinburgh,France,medium_airport,Edinburgh Airport,FR,GB-ENG,London,EDI,ASP,summer,low
freq,731,731,44181,53891,731,43572,9494,2921,731,86070,27174,36034


## 3. Missing Value Analysis

We count the number of missing values in each column and determine the percentage.
This drives the cleaning strategy: columns with a small percentage of nulls can be filled;
rows that are fundamentally incomplete (no airport metadata) should be dropped.

In [6]:
#So that I can go through all the columns and see which columns have missing values, and calculate their number, percentage, and type.
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "dtype": df.dtypes,
})

# Only show columns that have at least one missing value
missing = missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)
print(f"Columns with missing values: {len(missing)} out of {df.shape[1]}\n")
missing

Columns with missing values: 15 out of 37



,missing_count,missing_pct,dtype
iata_code,6720,6.26,str
latitude_deg,3814,3.56,float64
type,3814,3.56,str
longitude_deg,3814,3.56,float64
elevation_ft,3814,3.56,float64
iso_country,3814,3.56,str
name,3814,3.56,str
scheduled_service,3814,3.56,float64
runway_count,3814,3.56,float64
municipality,3814,3.56,str


### Missing Value Strategy

After inspecting the missing values, we apply the following strategy:

| Column Group | Issue                                                                              | Action | Rationale |
|---|------------------------------------------------------------------------------------|---|---|
| `type`, `name`, `latitude_deg`, `longitude_deg`, `elevation_ft`, `iso_country`, `iso_region`, `municipality`, `scheduled_service`, `runway_count`, `max_runway_length_ft`, `primary_surface` | ~3.6% --> airports in EUROCONTROL but not in OurAirports (small regional airports) | **Drop rows** | These rows have no airport metadata at all — filling would be pure guesswork and would distort the model |
| `iata_code` | ~6.3% --> some airports only have ICAO codes                                       | **Fill with "NONE"** | IATA code is an identifier, not a model feature; we keep the row |
| `avg_atfm_delay_min`, `pct_delayed_15` | ~1.9% --> division by zero where `arrivals == 0`                                   | **Dropped automatically** when we drop metadata-missing rows; any survivors filled with 0 | Zero arrivals means zero delay |
| `municipality` | Small number                                                                       | **Fill with "Unknown"** | Keep the row, mark the gap |

s)


## 4. Applying the Missing Value Treatment

In [7]:
rows_before = len(df)

# Step 1: Drop rows where airport metadata is missing entirely
# The 'type' column is NaN only when the EUROCONTROL airport had no OurAirports match.
df = df.dropna(subset=["type"]).copy()
rows_after_meta = len(df)
print(f"Step 1 — Dropped rows with no airport metadata: {rows_before - rows_after_meta:,}")

# Step 2: Fill remaining NaN in avg_atfm_delay_min and pct_delayed_15 with 0
#         (these come from days with 0 arrivals — no arrivals means no delay)
n_delay_null = df["avg_atfm_delay_min"].isnull().sum()
df["avg_atfm_delay_min"] = df["avg_atfm_delay_min"].fillna(0.0)
df["pct_delayed_15"] = df["pct_delayed_15"].fillna(0.0)
print(f"Step 2 — Filled {n_delay_null:,} NaN in delay columns with 0")

# Step 3: Fill missing iata_code with "NONE"
n_iata_null = df["iata_code"].isnull().sum()
df["iata_code"] = df["iata_code"].fillna("NONE")
print(f"Step 3 — Filled {n_iata_null:,} NaN in iata_code with 'NONE'")

# Step 4: Fill missing municipality with "Unknown"
n_muni_null = df["municipality"].isnull().sum()
df["municipality"] = df["municipality"].fillna("Unknown")
print(f"Step 4 — Filled {n_muni_null:,} NaN in municipality with 'Unknown'")

print(f"\nRows remaining: {len(df):,}  (dropped {rows_before - len(df):,} total)")
print(f"Remaining NaN:  {df.isnull().sum().sum()}")

Step 1 — Dropped rows with no airport metadata: 3,814
Step 2 — Filled 1,591 NaN in delay columns with 0
Step 3 — Filled 2,906 NaN in iata_code with 'NONE'
Step 4 — Filled 0 NaN in municipality with 'Unknown'

Rows remaining: 103,466  (dropped 3,814 total)
Remaining NaN:  0


## 5. Duplicate Detection and Removal

Each of the (airport, date) keys must be unique, with no duplication. If duplicates exist, we retain the first piece of information that appeared.

In [8]:
# Check for exact row-level duplicates
exact_dups = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups}")

# Check for logical duplicates (same airport + date)
logical_dups = df.duplicated(subset=["icao_code", "date"]).sum()
print(f"Logical duplicates (same airport + date): {logical_dups}")
#If there are duplicates, delete them and keep the first row.
if logical_dups > 0:
    df = df.drop_duplicates(subset=["icao_code", "date"], keep="first")
    print(f"Removed {logical_dups} logical duplicates") #If there are no repetitions, then of course there are no repetitions.
else:
    print("No duplicates found — dataset is clean on this front.")

print(f"Shape after dedup: {df.shape}")

Exact duplicate rows: 0
Logical duplicates (same airport + date): 0
No duplicates found — dataset is clean on this front.
Shape after dedup: (103466, 37)


## 6. Data Type Corrections

Several columns were read as `float64` because of the presence of NaN values before the cleanup.
Now that the NaN has been removed, we can convert them to their correct integer types.
We also ensure that the classification columns are stored as strings.

In [9]:
print("Data types BEFORE correction:")
print(df[["scheduled_service", "runway_count", "max_runway_length_ft", "elevation_ft"]].dtypes)
print()

# Float-to-int conversions (these were float only because of NaN)
df["scheduled_service"] = df["scheduled_service"].astype(int)
df["runway_count"] = df["runway_count"].astype(int)

# elevation_ft and max_runway_length_ft stay as float (genuine continuous measurements)

# Ensure categorical columns are string type
cat_cols = ["icao_code", "apt_name", "country", "type", "name",
            "iso_country", "iso_region", "municipality", "iata_code",
            "primary_surface", "season", "traffic_volume_class"]
for col in cat_cols:
    df[col] = df[col].astype(str)

# Ensure binary flags are int
for col in ["is_weekend", "is_summer", "hub_flag"]:
    df[col] = df[col].astype(int)

print("Data types AFTER correction:")
print(df[["scheduled_service", "runway_count", "max_runway_length_ft", "elevation_ft"]].dtypes)
print()
print("All dtypes:")
print(df.dtypes)

Data types BEFORE correction:
scheduled_service       float64
runway_count            float64
max_runway_length_ft    float64
elevation_ft            float64
dtype: object

Data types AFTER correction:
scheduled_service         int64
runway_count              int64
max_runway_length_ft    float64
elevation_ft            float64
dtype: object

All dtypes:
icao_code                          str
apt_name                           str
country                            str
date                    datetime64[us]
arrivals                         int64
total_atfm_delay_min             int64
flights_with_delay               int64
flights_delayed_15               int64
avg_atfm_delay_min             float64
pct_delayed_15                 float64
delay_weather_min                int64
delay_capacity_min               int64
delay_staffing_min               int64
departures                       int64
total_movements                  int64
type                               str
name               

## 7. Outlier Detection and Capping

We use two different strategies to determine extreme values based on column distribution:

- **Delay columns** (high zero values): IQR is zero because more than 75% of values are 0.Therefore, determining extreme values based on the IQR would eliminate all non-zero values. Instead, we cap at the **99th percentile** of non-zero values.
- **Traffic and physical columns** (continuous, high non-zero values): Standard **IQR** upper fence (`Q3 + 1.5 × IQR`).

In both cases, we determine extreme values instead of deleting rows, thus preserving the row count.



In [10]:
def cap_outliers_iqr(series, name):
    """Cap values above Q3 + 1.5*IQR. For columns with meaningful spread."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR
    n_outliers = (series > upper_fence).sum()
    pct = n_outliers / len(series) * 100
    capped = series.clip(upper=upper_fence)
    print(f"  {name:<30} IQR method  Fence={upper_fence:>10.1f}  "
          f"Capped={n_outliers:>5,} ({pct:.1f}%)")
    return capped


def cap_outliers_percentile(series, name, pctile=99):
    """Cap values above the Nth percentile of non-zero values.
    Used for zero-inflated delay columns where IQR = 0."""
    nonzero = series[series > 0]
    if len(nonzero) == 0:
        print(f"  {name:<30} percentile  (all zeros — skipped)")
        return series
    upper_fence = nonzero.quantile(pctile / 100)
    n_outliers = (series > upper_fence).sum()
    pct = n_outliers / len(series) * 100
    capped = series.clip(upper=upper_fence)
    print(f"  {name:<30} P{pctile} (non-zero)  Fence={upper_fence:>10.1f}  "
          f"Capped={n_outliers:>5,} ({pct:.1f}%)")
    return capped


# --- Zero-inflated delay columns: use 99th percentile of non-zero values ---
delay_cols = [
    "total_atfm_delay_min",
    "avg_atfm_delay_min",
    "pct_delayed_15",
    "delay_weather_min",
    "delay_capacity_min",
    "delay_staffing_min",
]

print("Outlier capping — delay columns (99th percentile of non-zero values):\n")
for col in delay_cols:
    df[col] = cap_outliers_percentile(df[col], col, pctile=99)

# --- Continuous columns: use standard IQR ---
iqr_cols = [
    "arrivals",
    "departures",
    "total_movements",
    "elevation_ft",
]

print("\nOutlier capping — traffic/physical columns (IQR method):\n")
for col in iqr_cols:
    df[col] = cap_outliers_iqr(df[col], col)

print(f"\nShape after capping: {df.shape}  (no rows removed — values capped in place)")

Outlier capping — delay columns (99th percentile of non-zero values):

  total_atfm_delay_min           P99 (non-zero)  Fence=    8835.0  Capped=   97 (0.1%)
  avg_atfm_delay_min             P99 (non-zero)  Fence=      31.0  Capped=   97 (0.1%)
  pct_delayed_15                 P99 (non-zero)  Fence=      51.9  Capped=   84 (0.1%)
  delay_weather_min              P99 (non-zero)  Fence=   14782.0  Capped=   28 (0.0%)
  delay_capacity_min             P99 (non-zero)  Fence=    1764.0  Capped=   18 (0.0%)
  delay_staffing_min             P99 (non-zero)  Fence=    4094.3  Capped=   19 (0.0%)

Outlier capping — traffic/physical columns (IQR method):

  arrivals                       IQR method  Fence=     211.0  Capped=10,155 (9.8%)
  departures                     IQR method  Fence=     211.0  Capped=10,147 (9.8%)
  total_movements                IQR method  Fence=     423.5  Capped=10,140 (9.8%)
  elevation_ft                   IQR method  Fence=    1477.5  Capped=11,541 (11.2%)

Shape afte

In [11]:
# Verify the outlier capping: compare before vs. after statistics
print("Post-capping summary of key delay columns:\n")
df[["avg_atfm_delay_min", "pct_delayed_15", "total_atfm_delay_min",
    "arrivals", "departures"]].describe().round(2)

Post-capping summary of key delay columns:



,avg_atfm_delay_min,pct_delayed_15,total_atfm_delay_min,arrivals,departures
count,103466.00,103466.00,103466.00,103466.00,103466.00
mean,0.29,0.64,69.00,55.51,55.41
std,1.76,3.56,476.45,69.51,69.52
min,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,6.00,6.00
50%,0.00,0.00,0.00,18.00,18.00
75%,0.00,0.00,0.00,88.00,88.00
max,30.95,51.86,8835.04,211.00,211.00


## 8. Ordinal Encoding

For modeling purposes, ordinal class attributes require numerical representation.
We encode `season` and `traffic_volume_class` with meaningful ordinal mappings.
Nominal categoricals (country, type, surface) will be handled in the modeling notebook
with one-hot encoding if needed.

In [12]:
# Convert season values into numbers

season_order = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
df["season_encoded"] = df["season"].map(season_order)

# Convert traffic volume levels into numbers
traffic_order = {"low": 0, "medium": 1, "high": 2}
df["traffic_volume_encoded"] = df["traffic_volume_class"].map(traffic_order)


# Show the new encoded values
print("Season encoding:")
print(df[["season", "season_encoded"]].drop_duplicates().sort_values("season_encoded").to_string(index=False))
print()
print("Traffic volume encoding:")
print(df[["traffic_volume_class", "traffic_volume_encoded"]].drop_duplicates()
      .sort_values("traffic_volume_encoded").to_string(index=False))
print()
print(f"New columns added: season_encoded, traffic_volume_encoded")
print(f"Shape: {df.shape}")

Season encoding:
season  season_encoded
winter               0
spring               1
summer               2
autumn               3

Traffic volume encoding:
traffic_volume_class  traffic_volume_encoded
                 low                       0
              medium                       1
                high                       2

New columns added: season_encoded, traffic_volume_encoded
Shape: (103466, 39)


## 9. Surface Type Standardisation

The `primary_surface` column has abbreviations and mixed naming conventions from OurAirports.
We group them into a small number of standard categories to reduce cardinality (the number of values).

In [13]:
# Check current unique values
print("Surface types before standardisation:")
print(df["primary_surface"].value_counts())
print()

# Map to standard categories
surface_map = {
    "ASP":      "asphalt",
    "ASPH":     "asphalt",
    "Asphalt":  "asphalt",
    "asphalt":  "asphalt",       # lowercase variant in source data
    "BIT":      "asphalt",       # Bitumen = asphalt variant
    "PEM":      "asphalt",       # PEM = Paved/Engineered Material — closest to asphalt
    "CON":      "concrete",
    "Concrete": "concrete",
    "GRS":      "grass",
    "Grass":    "grass",
    "GRASS":    "grass",         # uppercase variant in source data
    "GRE":      "gravel",
}

df["primary_surface"] = df["primary_surface"].map(surface_map).fillna("other")

print("Surface types after standardisation:")
print(df["primary_surface"].value_counts())

Surface types before standardisation:
primary_surface
ASP        86070
CON        11623
ASPH        2372
GRASS        731
asphalt      714
PEM          675
hard         664
GRS          617
Name: count, dtype: int64

Surface types after standardisation:
primary_surface
asphalt     89831
concrete    11623
grass        1348
other         664
Name: count, dtype: int64


## 10. Final Validation

Before saving, we verify that the clean dataset satisfies all project requirements:
- No remaining null values
- No duplicate rows
- At least 1,000 rows
- At least 15 features (excluding identifiers and target)
- Exactly 5 countries

In [14]:
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

# We check the data before final saving.
# number of rows
n_rows = len(df)
# number of coulmn
n_cols = df.shape[1]
# number of data equal null
n_nulls = df.isnull().sum().sum()
# number of rows duplicate
n_dups = df.duplicated().sum()
#number of diffrent country
n_countries = df["country"].nunique()

checks = [
    ("Rows >= 1,000",       n_rows >= 1000,     f"{n_rows:,}"),
    ("Columns >= 15",       n_cols >= 15,        f"{n_cols} (incl. 2 encoded)"),
    ("Countries == 5",      n_countries == 5,    f"{n_countries}"),
    ("No null values",      n_nulls == 0,        f"{n_nulls}"),
    ("No duplicate rows",   n_dups == 0,         f"{n_dups}"),
]

all_pass = True
for label, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  {label:<25} {value:<25} {status}")

print("=" * 60)
if all_pass:
    print("All checks passed.")
else:
    print("SOME CHECKS FAILED — review the issues above.")

# Hard assertions
assert n_nulls == 0, f"Still {n_nulls} null values!"
assert n_dups == 0, f"Still {n_dups} duplicates!"
assert n_rows >= 1000
assert n_cols >= 15
assert n_countries == 5

FINAL VALIDATION
  Rows >= 1,000             103,466                   PASS
  Columns >= 15             39 (incl. 2 encoded)      PASS
  Countries == 5            5                         PASS
  No null values            0                         PASS
  No duplicate rows         0                         PASS
All checks passed.


In [16]:
country_summary = pd.DataFrame({
    "Number of Records": df["country"].value_counts(),
    "Number of Airports": df.groupby("country")["icao_code"].nunique()
})

country_summary = country_summary.reset_index()
country_summary.columns = ["Country", "Number of Records", "Number of Airports"]

country_summary

,Country,Number of Records,Number of Airports
0,France,43572,62
1,Germany,10941,15
2,Netherlands,3557,5
3,Spain,31521,46
4,United Kingdom,13875,19


## 11. Save the Clean Dataset

In [9]:
df.to_csv(OUTPUT_PATH, index=False)
#save data cleaning the file
size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
#print path file
print(f"Saved: {OUTPUT_PATH}")
#print size the file
print(f"Size:  {size_mb:.1f} MB")
#print number of row and coulmn
print(f"Shape: {df.shape}")

Saved: ..\data\processed\flight_delays_clean.csv
Size:  20.8 MB
Shape: (103466, 37)


## 12. Cleaning Summary

The table below documents every cleaning action taken, why it was needed, and how many rows
were affected. This serves as the reference for report section 3.3.

In [13]:
summary = pd.DataFrame([
    {"Step": "1. Drop unmatched airports",
     "Columns Affected": "type, name, lat, lon, elevation, iso_country, iso_region, municipality, scheduled_service, runway_count, max_runway_length_ft, primary_surface",
     "Issue": "3.6% of rows had no OurAirports match (small regional airports)",
     "Action": "Dropped rows where type is NaN",
     "Rows Affected": "3,814 rows dropped"},

    {"Step": "2. Fill delay NaN",
     "Columns Affected": "avg_atfm_delay_min, pct_delayed_15",
     "Issue": "Division by zero where arrivals = 0",
     "Action": "Filled with 0.0 (no arrivals = no delay)",
     "Rows Affected": "1,591 rows filled"},

    {"Step": "3. Fill iata_code NaN",
     "Columns Affected": "iata_code",
     "Issue": "Some airports only have ICAO codes",
     "Action": "Filled with 'NONE'",
     "Rows Affected": "2,906 rows filled"},

    {"Step": "4. Fill municipality NaN",
     "Columns Affected": "municipality",
     "Issue": "Small number of airports missing city name",
     "Action": "Filled with 'Unknown'",
     "Rows Affected": "0 rows (none remaining after Step 1)"},

    {"Step": "5. Remove duplicates",
     "Columns Affected": "All",
     "Issue": "Check for exact and logical (airport+date) duplicates",
     "Action": "Drop duplicates, keep first",
     "Rows Affected": "0 (none found)"},

    {"Step": "6. Fix data types",
     "Columns Affected": "scheduled_service, runway_count, binary flags, categoricals",
     "Issue": "Float columns that should be int; inconsistent string types",
     "Action": "Cast to correct dtypes",
     "Rows Affected": "0 (in-place conversion)"},

    {"Step": "7. Cap outliers",
     "Columns Affected": "Delay columns + arrivals, departures, total_movements, elevation_ft",
     "Issue": "Extreme right-tail values (storm days, mega-hubs)",
     "Action": "Delay cols: P99 of non-zero values; traffic/physical cols: IQR upper fence",
     "Rows Affected": "Varies per column (values clipped, no rows dropped)"},

    {"Step": "8. Ordinal encoding",
     "Columns Affected": "season -> season_encoded, traffic_volume_class -> traffic_volume_encoded",
     "Issue": "Categorical features need numeric form for modelling",
     "Action": "Mapped to ordinal integers (0, 1, 2, 3)",
     "Rows Affected": "All rows (2 new columns added)"},

    {"Step": "9. Surface standardisation",
     "Columns Affected": "primary_surface",
     "Issue": "Mixed abbreviations and case variants (ASP, ASPH, Asphalt, asphalt, CON, GRASS, GRS, etc.)",
     "Action": "Mapped to standard names: asphalt, concrete, grass, gravel, other",
     "Rows Affected": "All rows"},
])

# Display the summary nicely
summary.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,Step,Columns Affected,Issue,Action,Rows Affected
0,1. Drop unmatched airports,"type, name, lat, lon, elevation, iso_country, iso_region, municipality, scheduled_service, runway_count, max_runway_length_ft, primary_surface",3.6% of rows had no OurAirports match (small regional airports),Dropped rows where type is NaN,"3,814 rows dropped"
1,2. Fill delay NaN,"avg_atfm_delay_min, pct_delayed_15",Division by zero where arrivals = 0,Filled with 0.0 (no arrivals = no delay),"1,591 rows filled"
2,3. Fill iata_code NaN,iata_code,Some airports only have ICAO codes,Filled with 'NONE',"2,906 rows filled"
3,4. Fill municipality NaN,municipality,Small number of airports missing city name,Filled with 'Unknown',0 rows (none remaining after Step 1)
4,5. Remove duplicates,All,Check for exact and logical (airport+date) duplicates,"Drop duplicates, keep first",0 (none found)
5,6. Fix data types,"scheduled_service, runway_count, binary flags, categoricals",Float columns that should be int; inconsistent string types,Cast to correct dtypes,0 (in-place conversion)
6,7. Cap outliers,"Delay columns + arrivals, departures, total_movements, elevation_ft","Extreme right-tail values (storm days, mega-hubs)",Delay cols: P99 of non-zero values; traffic/physical cols: IQR upper fence,"Varies per column (values clipped, no rows dropped)"
7,8. Ordinal encoding,"season -> season_encoded, traffic_volume_class -> traffic_volume_encoded",Categorical features need numeric form for modelling,"Mapped to ordinal integers (0, 1, 2, 3)",All rows (2 new columns added)
8,9. Surface standardisation,primary_surface,"Mixed abbreviations and case variants (ASP, ASPH, Asphalt, asphalt, CON, GRASS, GRS, etc.)","Mapped to standard names: asphalt, concrete, grass, gravel, other",All rows


## 13. Clean Dataset Column Reference

In [11]:
print(f"{'#':<4} {'Column':<30} {'Dtype':<15} {'Non-Null':<12} {'Sample'}")
print("-" * 85)
for i, col in enumerate(df.columns, 1):
    dtype = str(df[col].dtype)
    non_null = df[col].notna().sum()
    sample = df[col].dropna().iloc[0] if df[col].notna().any() else "NaN"
    # Truncate long sample values
    sample_str = str(sample)
    if len(sample_str) > 25:
        sample_str = sample_str[:22] + "..."
    print(f"{i:<4} {col:<30} {dtype:<15} {non_null:<12} {sample_str}")

#    Column                         Dtype           Non-Null     Sample
-------------------------------------------------------------------------------------
1    icao_code                      str             103466       EGPH
2    apt_name                       str             103466       Edinburgh
3    country                        str             103466       United Kingdom
4    date                           datetime64[us]  103466       2023-01-01 00:00:00
5    arrivals                       int64           103466       105
6    total_atfm_delay_min           int64           103466       77
7    flights_with_delay             int64           103466       4
8    flights_delayed_15             int64           103466       2
9    avg_atfm_delay_min             float64         103466       0.7333333333333333
10   pct_delayed_15                 float64         103466       1.9047619047619049
11   delay_weather_min              int64           103466       0
12   delay_capacity_min   

---
**Next step:** `03_eda.ipynb` — Exploratory Data Analysis and Visualisations